# Explore the movie data

This notebook explores cached films from the current candidate list. The data profiles include every matched film. The numeric and category outcome comparisons use dated films marked Released, across all years. Notebook 04 uses these release rules and runs the licensing comparison for both windows.


In [1]:
from IPython.display import display
import pandas as pd

from streamvault.cache import iter_cached_records, match_cached_candidates
from streamvault.config import CONFIG
from streamvault.diagnostics import raise_for_failed_checks, run_data_checks
from streamvault.load import iter_movie_id_rows
from streamvault.movie_quality import film_quality_issues, financial_inputs, rating_inputs
from streamvault.normalize import normalization_checks, normalize_movie_records

cache = match_cached_candidates(
    iter_cached_records(CONFIG.cache_file),
    iter_movie_id_rows(CONFIG.source_file),
)
stage = "Loaded current candidates"
check_report = run_data_checks(stage, {
    "Current candidate IDs are unique in the cache": cache.uniqueness_check(),
})
display(check_report)
raise_for_failed_checks(stage, check_report)
records = list(cache.records.values())
tables = normalize_movie_records(records)
movies = tables.movies
stage = "After normalization"
check_report = normalization_checks(stage, tables, set(cache.records))
display(check_report)
raise_for_failed_checks(stage, check_report)

issues = film_quality_issues(movies)
if not issues.empty:
    print("Film data issues to review")
    display(issues)

# Keep the full list for profiling, including films with missing release details.
release_dates = movies.get(
    "release_date", pd.Series(pd.NaT, index=movies.index, dtype="datetime64[ns]")
)
status = movies.get("status", pd.Series(pd.NA, index=movies.index, dtype="string"))
dated = release_dates.notna()
released = status.eq("Released").fillna(False)
all_years = dated & released
since_2000 = all_years & release_dates.ge(pd.Timestamp("2000-01-01"))

# Count undated films first so each exclusion has only one reason.
film_counts = pd.Series({
    "Matched films on the current list": len(movies),
    "Missing release date": int((~dated).sum()),
    "Dated, missing release status": int((dated & status.isna()).sum()),
    "Dated, marked with another status": int((dated & status.notna() & ~released).sum()),
    "Dated releases across all years": int(all_years.sum()),
    "Dated releases from 2000 onward": int(since_2000.sum()),
}, name="films")
display(film_counts.to_frame())
if not records:
    print("No cached movies yet. Run `uv run streamvault` first.")

# Format to 4 decimal places with comma separators (e.g., 108,446,000.00)
pd.set_option('display.float_format', '{:,.4f}'.format)

,check,status,details
0,Current candidate IDs are unique in the cache,PASS,112 candidates already cached. 0 repeated IDs....


,check,status,details
0,Movie IDs are present and unique,PASS,112 distinct IDs in 112 rows
1,Movie IDs match the resolved candidates,PASS,"112 expected, 112 found. Missing examples: []...."
2,Genre and credit links resolve,PASS,Every link points to an existing film and refe...
3,Collection join preserves films and resolves s...,PASS,112 rows from 112 films. Unmatched example mov...


,films
Matched films on the current list,112
Missing release date,0
"Dated, missing release status",0
"Dated, marked with another status",0
Dated releases across all years,112
Dated releases from 2000 onward,71


## 1. Numeric movie fields and unusual values

Use quantiles and interquartile-range (IQR) bounds to locate values worth checking. These bounds are exploratory, not universal validity rules; budget and revenue may be zero or absent, for example, and an extreme value may be legitimate.


In [2]:
identifier_columns = {"movie_id", "collection_id"}
numeric_movie_columns = [
    column
    for column in movies.select_dtypes(include="number").columns
    if column not in identifier_columns
]

if numeric_movie_columns:
    numeric_summary = movies[numeric_movie_columns].describe(
        percentiles=[0.01, 0.25, 0.50, 0.75, 0.90, 0.99]
    ).T
    display(numeric_summary)
else:
    print("No numeric movie measures are available.")

outlier_rows = []
for column in numeric_movie_columns:
    values = pd.to_numeric(movies[column], errors="coerce")
    valid_values = values.dropna()
    if valid_values.empty:
        continue

    q1 = valid_values.quantile(0.25)
    q3 = valid_values.quantile(0.75)
    iqr = q3 - q1
    lower_bound = q1 - 1.5 * iqr
    upper_bound = q3 + 1.5 * iqr
    unusual = values.lt(lower_bound) | values.gt(upper_bound)
    examples = movies.loc[unusual, ["movie_id", column]].head(5)

    outlier_rows.append(
        {
            "field": column,
            "iqr_lower_bound": lower_bound,
            "iqr_upper_bound": upper_bound,
            "values_outside_iqr": int(unusual.sum()),
            "negative_values": int(valid_values.lt(0).sum()),
            "example_movie_values": [
                f"{row.movie_id}: {row_value}"
                for row, row_value in zip(
                    examples.itertuples(index=False),
                    examples[column],
                    strict=True,
                )
            ],
        }
    )

outlier_review = pd.DataFrame(outlier_rows)
display(outlier_review)


,count,mean,std,min,1%,25%,50%,75%,90%,99%,max
budget,112.0000,"108,445,982.1429","82,788,177.8615","1,200,000.0000","4,665,000.0000","38,087,500.0000","91,000,000.0000","150,000,000.0000","229,000,000.0000","349,840,000.0000","379,000,000.0000"
popularity,112.0000,19.6574,13.4498,0.4583,1.0591,10.3549,16.3064,23.9677,36.8038,65.7337,89.1177
revenue,112.0000,"599,658,765.7411","436,246,692.9360","7,390,108.0000","29,502,907.4100","295,346,340.0000","519,563,017.5000","872,698,393.5000","1,057,280,933.6000","1,993,719,091.3600","2,942,649,778.0000"
runtime,112.0000,130.4911,25.5368,81.0000,81.8800,115.0000,128.5000,143.2500,164.7000,200.3400,202.0000
vote_average,112.0000,7.5887,0.7135,5.7520,6.0359,7.2012,7.7200,8.1645,8.4510,8.6753,8.7340
vote_count,112.0000,"17,538.3214","8,270.0406","4,136.0000","5,959.5000","11,039.0000","15,841.0000","22,821.5000","29,265.2000","40,713.8500","41,925.0000"


,field,iqr_lower_bound,iqr_upper_bound,values_outside_iqr,negative_values,example_movie_values
0,budget,"-129,781,250.0000","317,868,750.0000",2,0,"[299534: 356000000, 1865: 379000000]"
1,popularity,-10.0644,44.3869,6,0,"[155: 66.8926, 278: 45.4678, 299534: 89.1177, ..."
2,revenue,"-570,681,740.2500","1,738,726,473.7500",2,0,"[299534: 2942649778, 299536: 2052415039]"
3,runtime,72.6250,185.6250,3,0,"[122: 201, 240: 202, 424: 195]"
4,vote_average,5.7564,9.6094,1,0,[41513: 5.752]
5,vote_count,"-6,634.7500","40,495.2500",2,0,"[27205: 40754, 157336: 41925]"


### 1.1. Investigating the unusual values
IQR flags are screening candidates, not validity rules. The checks below separate missing or negative values from plausible tail observations, and compare each candidate with related fields before drawing a preliminary conclusion.

In [3]:
investigation_movies = movies.copy()
investigation_movies["release_year"] = release_dates.dt.year

unusual_details = []
for column in numeric_movie_columns:
    values = pd.to_numeric(investigation_movies[column], errors="coerce")
    valid_values = values.dropna()
    if valid_values.empty:
        continue

    q1 = valid_values.quantile(0.25)
    q3 = valid_values.quantile(0.75)
    iqr = q3 - q1
    lower_bound = q1 - 1.5 * iqr
    upper_bound = q3 + 1.5 * iqr
    unusual = values.lt(lower_bound) | values.gt(upper_bound)

    for row in investigation_movies.loc[unusual].itertuples(index=False):
        value = getattr(row, column)
        unusual_details.append(
            {
                "field": column,
                "movie_id": row.movie_id,
                "title": row.title if "title" in investigation_movies.columns else pd.NA,
                "release_year": (
                    getattr(row, "release_year")
                    if "release_year" in investigation_movies.columns
                    else pd.NA
                ),
                "value": value,
                "tail": "low" if value < lower_bound else "high",
                "iqr_lower_bound": lower_bound,
                "iqr_upper_bound": upper_bound,
            }
        )

# Keep these columns even when the screen finds no unusual films.
unusual_detail_table = pd.DataFrame(unusual_details, columns=[
    "field", "movie_id", "title", "release_year", "value", "tail",
    "iqr_lower_bound", "iqr_upper_bound",
])
display(unusual_detail_table)

,field,movie_id,title,release_year,value,tail,iqr_lower_bound,iqr_upper_bound
0,budget,299534,Avengers: Endgame,2019,"356,000,000.0000",high,"-129,781,250.0000","317,868,750.0000"
1,budget,1865,Pirates of the Caribbean: On Stranger Tides,2011,"379,000,000.0000",high,"-129,781,250.0000","317,868,750.0000"
2,popularity,155,The Dark Knight,2008,66.8926,high,-10.0644,44.3869
3,popularity,278,The Shawshank Redemption,1994,45.4678,high,-10.0644,44.3869
4,popularity,299534,Avengers: Endgame,2019,89.1177,high,-10.0644,44.3869
5,popularity,27205,Inception,2010,45.4139,high,-10.0644,44.3869
6,popularity,299536,Avengers: Infinity War,2018,49.4655,high,-10.0644,44.3869
7,popularity,157336,Interstellar,2014,56.3569,high,-10.0644,44.3869
8,revenue,299534,Avengers: Endgame,2019,"2,942,649,778.0000",high,"-570,681,740.2500","1,738,726,473.7500"
9,revenue,299536,Avengers: Infinity War,2018,"2,052,415,039.0000",high,"-570,681,740.2500","1,738,726,473.7500"


#### 1.1.1. Financial values: budget and revenue
These checks look for contradictory signs or scale errors, not merely large nominal amounts. A large budget or revenue is possible for a blockbuster; a revenue below zero, a negative budget, or a ratio that lacks a sensible explanation would be a stronger anomaly signal.

In [4]:
financial_columns = [column for column in ("budget", "revenue") if column in investigation_movies.columns]
if financial_columns:
    financial = investigation_movies[["movie_id", "title", "release_year", *financial_columns]].copy()
    for column in financial_columns:
        financial[column] = pd.to_numeric(financial[column], errors="coerce")
        financial[f"{column}_missing"] = financial[column].isna()
        financial[f"{column}_negative"] = financial[column].lt(0)
        financial[f"{column}_zero"] = financial[column].eq(0)

    if "budget" in financial and "revenue" in financial:
        # Keep the original amounts visible above. Use only usable pairs for the ratio.
        prepared = financial_inputs(financial)
        financial["revenue_budget_ratio"] = prepared["revenue"] / prepared["budget"]

    print("Financial integrity checks:")
    for column in financial_columns:
        values = pd.to_numeric(financial[column], errors="coerce")
        print(
            f"{column}: missing={values.isna().sum()}, zero={values.eq(0).sum()}, "
            f"negative={values.lt(0).sum()}"
        )

    financial_extreme_ids = set(
        unusual_detail_table.loc[
            unusual_detail_table["field"].isin(financial_columns), "movie_id"
        ]
    )
    if financial_extreme_ids:
        display(financial.loc[financial["movie_id"].isin(financial_extreme_ids)].sort_values("movie_id").reset_index(drop=True))

    if "revenue_budget_ratio" in financial:
        ratio_summary = financial["revenue_budget_ratio"].describe(percentiles=[0.25, 0.5, 0.75, 0.90, 0.99])
        print("Revenue/budget ratio summary for films with usable budget and gross:")
        display(ratio_summary.to_frame("ratio").T)

,movie_id,title,release_year,budget,revenue,budget_missing,budget_negative,budget_zero,revenue_missing,revenue_negative,revenue_zero,revenue_budget_ratio
0,1865,Pirates of the Caribbean: On Stranger Tides,2011,379000000,1046721266,False,False,False,False,False,False,2.7618
1,299534,Avengers: Endgame,2019,356000000,2942649778,False,False,False,False,False,False,8.2659
2,299536,Avengers: Infinity War,2018,300000000,2052415039,False,False,False,False,False,False,6.8414


,count,mean,std,min,25%,50%,75%,90%,99%,max
ratio,112.0000,8.5036,10.3771,1.1337,3.7969,5.3741,8.8502,14.5937,64.3331,70.4907


#### 1.1.2. Runtime values
The IQR high runtime candidates are reviewed with neighboring runtimes and release context. Runtime units, missing values, and a clearly implausible duration would be stronger anomaly evidence than being above a percentile bound.

In [5]:
if "runtime" in investigation_movies.columns:
    runtimes = pd.to_numeric(investigation_movies["runtime"], errors="coerce")
    print(
        f"runtime: missing={runtimes.isna().sum()}, zero={runtimes.eq(0).sum()}, "
        f"negative={runtimes.lt(0).sum()}"
    )
    runtime_extremes = unusual_detail_table.loc[unusual_detail_table["field"] == "runtime"]
    if not runtime_extremes.empty:
        display(
            investigation_movies.loc[
                investigation_movies["movie_id"].isin(runtime_extremes["movie_id"]),
                ["movie_id", "title", "release_year", "runtime"],
            ].sort_values("runtime", ascending=False).reset_index(drop=True)
        )
    print("Longest runtimes overall:")
    display(
        investigation_movies.sort_values("runtime", ascending=False)[
            ["movie_id", "title", "release_year", "runtime", "vote_average"]
        ].head(12).reset_index(drop=True)
    )

,movie_id,title,release_year,runtime
0,240,The Godfather Part II,1974,202
1,122,The Lord of the Rings: The Return of the King,2003,201
2,424,Schindler's List,1993,195


,movie_id,title,release_year,runtime,vote_average
0,240,The Godfather Part II,1974,202,8.5730
1,122,The Lord of the Rings: The Return of the King,2003,201,8.5070
2,424,Schindler's List,1993,195,8.5700
3,299534,Avengers: Endgame,2019,181,8.2470
4,121,The Lord of the Rings: The Two Towers,2002,179,8.4330
5,120,The Lord of the Rings: The Fellowship of the Ring,2001,179,8.4510
6,238,The Godfather,1972,175,8.6880
7,111,Scarface,1983,170,8.1540
8,49051,The Hobbit: An Unexpected Journey,2012,169,7.3890
9,285,Pirates of the Caribbean: At World's End,2007,169,7.2810


#### 1.1.3. Popularity and vote metrics
Popularity, vote count, and vote average measure different things. A high-popularity movie may have strong audience engagement; it is not necessarily duplicated or erroneous. Likewise, a low rating can be plausible when supported by nearby ratings and a meaningful vote count.

In [6]:
metric_columns = [
    column
    for column in ("popularity", "vote_count", "vote_average")
    if column in investigation_movies.columns
]
for column in metric_columns:
    values = pd.to_numeric(investigation_movies[column], errors="coerce")
    print(
        f"{column}: missing={values.isna().sum()}, zero={values.eq(0).sum()}, "
        f"negative={values.lt(0).sum()}"
    )

if "popularity" in investigation_movies:
    print("Highest popularity candidates:")
    display(
        investigation_movies.sort_values("popularity", ascending=False)[
            ["movie_id", "title", "release_year", "popularity", "vote_count", "vote_average"]
        ].head(12).reset_index(drop=True)
    )

if "vote_average" in investigation_movies:
    print("Lowest vote averages:")
    display(
        investigation_movies.sort_values("vote_average", ascending=True)[
            ["movie_id", "title", "release_year", "vote_average", "vote_count", "popularity"]
        ].head(15).reset_index(drop=True)
    )

if "vote_count" in investigation_movies:
    print("Highest vote-count candidates:")
    display(
        investigation_movies.sort_values("vote_count", ascending=False)[
            ["movie_id", "title", "release_year", "vote_count", "popularity", "vote_average"]
        ].head(12).reset_index(drop=True)
    )


,movie_id,title,release_year,popularity,vote_count,vote_average
0,299534,Avengers: Endgame,2019,89.1177,29384,8.2470
1,155,The Dark Knight,2008,66.8926,37405,8.5400
2,157336,Interstellar,2014,56.3569,41925,8.4930
3,299536,Avengers: Infinity War,2018,49.4655,33418,8.2470
4,278,The Shawshank Redemption,1994,45.4678,31864,8.7340
5,27205,Inception,2010,45.4139,40754,8.4000
6,671,Harry Potter and the Philosopher's Stone,2001,43.3460,30644,7.9050
7,24428,The Avengers,2012,42.1051,40389,8.0850
8,238,The Godfather,1972,40.7425,23999,8.6880
9,120,The Lord of the Rings: The Fellowship of the Ring,2001,39.3976,28890,8.4510


,movie_id,title,release_year,vote_average,vote_count,popularity
0,41513,The Smurfs,2011,5.7520,4136,5.4347
1,217,Indiana Jones and the Kingdom of the Crystal S...,2008,6.0270,9386,7.9677
2,534,Terminator Salvation,2009,6.1080,7360,8.2923
3,955,Mission: Impossible II,2000,6.1400,7967,7.7691
4,8078,Alien Resurrection,1997,6.1700,5877,7.7192
5,331,Jurassic Park III,2001,6.1790,8195,0.4583
6,296,Terminator 3: Rise of the Machines,2003,6.2160,7641,9.4087
7,810,Shrek the Third,2007,6.3170,10138,11.1544
8,8077,Alien³,1992,6.4000,6722,7.7480
9,10192,Shrek Forever After,2010,6.4160,8237,13.7726


,movie_id,title,release_year,vote_count,popularity,vote_average
0,157336,Interstellar,2014,41925,56.3569,8.4930
1,27205,Inception,2010,40754,45.4139,8.4000
2,24428,The Avengers,2012,40389,42.1051,8.0850
3,155,The Dark Knight,2008,37405,66.8926,8.5400
4,299536,Avengers: Infinity War,2018,33418,49.4655,8.2470
5,550,Fight Club,1999,33362,33.3477,8.4000
6,278,The Shawshank Redemption,1994,31864,45.4678,8.7340
7,680,Pulp Fiction,1994,31350,29.9893,8.4800
8,13,Forrest Gump,1994,30907,28.9685,8.4620
9,671,Harry Potter and the Philosopher's Stone,2001,30644,43.3460,7.9050


#### 1.1.4. Examples from the original snapshot

These examples explain why some unusual values are plausible. Use the live tables above to see which films are flagged on a new run.

In [7]:
field_verdicts = [
    {
        "field": "budget",
        "flagged_movies": "Pirates of the Caribbean: On Stranger Tides; Avengers: Endgame",
        "evidence_of_normality": "Both values are positive and the films are known large-budget blockbusters; no negative or missing values occur.",
        "evidence_of_concern": "Nominal budgets are high and sit above the IQR upper bound.",
        "preliminary_conclusion": "Normal extreme",
    },
    {
        "field": "revenue",
        "flagged_movies": "Avengers: Endgame; Avengers: Infinity War",
        "evidence_of_normality": "Positive ratios and global blockbuster context support the high observations; no missing, zero, or negative values occur.",
        "evidence_of_concern": "Both values exceed the IQR upper bound and rank at the top of this cache.",
        "preliminary_conclusion": "Normal extreme",
    },
    {
        "field": "runtime",
        "flagged_movies": "The Godfather Part II; The Lord of the Rings: The Return of the King; Schindler's List",
        "evidence_of_normality": "All are positive, complete runtimes and are near other 195–202 minute films.",
        "evidence_of_concern": "Three runtimes exceed the IQR upper bound.",
        "preliminary_conclusion": "Normal extreme",
    },
    {
        "field": "popularity",
        "flagged_movies": "Avengers: Endgame; The Dark Knight; Interstellar; Avengers: Infinity War; Shawshank Redemption; Inception",
        "evidence_of_normality": "High values broadly align with high vote counts and no missing or negative values occur.",
        "evidence_of_concern": "Popularity is a cached/source-derived snapshot and can change independently of static film attributes.",
        "preliminary_conclusion": "Needs source verification",
    },
    {
        "field": "vote_average",
        "flagged_movies": "The Smurfs",
        "evidence_of_normality": "The 5.752 value is inside TMDB's 0–10 scale and is consistent with several nearby low-rated films.",
        "evidence_of_concern": "It is the sole IQR low tail value and has the smallest vote count among the cache.",
        "preliminary_conclusion": "Normal extreme",
    },
    {
        "field": "vote_count",
        "flagged_movies": "Interstellar; Inception",
        "evidence_of_normality": "Both are positive and high, with strong correlation to popularity and vote averages.",
        "evidence_of_concern": "Both exceed the IQR upper bound and are the cache's strongest vote-engagement observations.",
        "preliminary_conclusion": "Normal extreme",
    },
]
display(pd.DataFrame(field_verdicts))

,field,flagged_movies,evidence_of_normality,evidence_of_concern,preliminary_conclusion
0,budget,Pirates of the Caribbean: On Stranger Tides; A...,Both values are positive and the films are kno...,Nominal budgets are high and sit above the IQR...,Normal extreme
1,revenue,Avengers: Endgame; Avengers: Infinity War,Positive ratios and global blockbuster context...,Both values exceed the IQR upper bound and ran...,Normal extreme
2,runtime,The Godfather Part II; The Lord of the Rings: ...,"All are positive, complete runtimes and are ne...",Three runtimes exceed the IQR upper bound.,Normal extreme
3,popularity,Avengers: Endgame; The Dark Knight; Interstell...,High values broadly align with high vote count...,Popularity is a cached/source-derived snapshot...,Needs source verification
4,vote_average,The Smurfs,The 5.752 value is inside TMDB's 0–10 scale an...,It is the sole IQR low tail value and has the ...,Normal extreme
5,vote_count,Interstellar; Inception,"Both are positive and high, with strong correl...",Both exceed the IQR upper bound and are the ca...,Normal extreme


## 2. Relationships between numeric measures

Each comparison uses dated releases with usable values for both measures. Pearson measures how closely the values move in a straight line. Spearman measures how closely their rankings move together.

The main numbers include unusual but usable values. The last columns show what changes when films with unusually high or low values for either measure are left out. These are associations within the candidate list.

In [8]:
# Use the same usable inputs here and in the genre outcome tables below.
fields = ["movie_id", "collection_id", "budget", "revenue", "popularity", "vote_average", "vote_count"]
films = movies.loc[all_years].reindex(columns=fields).copy()
films["release_year"] = release_dates.loc[all_years].dt.year
ratings = rating_inputs(films)
films[["vote_average", "vote_count"]] = ratings[["vote_average", "vote_count"]]

# Apply the same budget and gross rules as the pipeline's quality report.
amounts = financial_inputs(films)
films[["budget", "revenue"]] = amounts[["budget", "revenue"]]
popularity = pd.to_numeric(films["popularity"].astype("string"), errors="coerce").astype(float)
films["popularity"] = popularity.where(popularity.between(0, float("inf"), inclusive="left"))

pairs = [
    ("Budget and gross", "budget", "revenue"),
    ("Popularity and vote count", "popularity", "vote_count"),
    ("Vote count and rating", "vote_count", "vote_average"),
    ("Gross and rating", "revenue", "vote_average"),
]
numeric_fields = ["budget", "revenue", "popularity", "vote_count", "vote_average"]
values = films[numeric_fields]
lower_quartile, upper_quartile = values.quantile(0.25), values.quantile(0.75)
spread = upper_quartile - lower_quartile
unusual = values.lt(lower_quartile - 1.5 * spread) | values.gt(upper_quartile + 1.5 * spread)

relationship_rows = []
for label, left, right in pairs:
    # Rank the same films for both fields so missing values cannot change one side's ranks.
    pair = values[[left, right]].dropna()
    trimmed = pair.loc[~unusual.loc[pair.index, [left, right]].any(axis=1)]
    relationship_rows.append({
        "Measures": label,
        "Films": len(pair),
        "Pearson": pair[left].corr(pair[right], min_periods=3),
        "Spearman": pair[left].rank().corr(pair[right].rank(), min_periods=3),
        "Films after omitting unusual values": len(trimmed),
        "Pearson after omission": trimmed[left].corr(trimmed[right], min_periods=3),
        "Spearman after omission": trimmed[left].rank().corr(trimmed[right].rank(), min_periods=3),
    })

relationship_summary = pd.DataFrame(relationship_rows)
display(relationship_summary.round(3))

,Measures,Films,Pearson,Spearman,Films after omitting unusual values,Pearson after omission,Spearman after omission
0,Budget and gross,112,0.7460,0.7530,109,0.7150,0.7340
1,Popularity and vote count,112,0.8300,0.8690,106,0.8540,0.8470
2,Vote count and rating,112,0.6590,0.6850,109,0.6430,0.6630
3,Gross and rating,112,-0.0210,-0.1440,109,-0.1240,-0.1840


## 3. Category analysis

The profiles describe the candidate list. The outcome tables use dated releases across all years. Notebook 04 uses both release windows for the licensing comparison and talent assessment.

### 3.1. Univariate profiles

Look at one field at a time to see which dates, labels, and links are present. The release-date and relationship profiles cover all matched films. The final profile shows category coverage for the dated releases used in the outcome comparisons.


#### 3.1.1. Release dates and common categories

Review coverage and the observed range before defining date rules. Frequency tables help identify dominant categories, spelling/label variants, or unexpected missing values.


In [9]:
if "release_date" in movies.columns:
    release_dates = movies["release_date"].dropna()
    print(f"Movies with a release date: {len(release_dates):,} / {len(movies):,}")
    if not release_dates.empty:
        print(f"Observed release-date range: {release_dates.min()} to {release_dates.max()}")
        release_year_counts = (
            release_dates.dt.year.value_counts().sort_index().rename("movie_count")
        )
        print("Earliest release years:")
        display(release_year_counts.head(15).to_frame())
        print("Latest release years:")
        display(release_year_counts.tail(15).to_frame())
else:
    print("No normalized release_date column is available.")

category_columns = [
    column
    for column in ("status", "original_language", "adult", "video")
    if column in movies.columns
]
category_rows = []
for column in category_columns:
    counts = movies[column].value_counts(dropna=False).head(15)
    for value, count in counts.items():
        category_rows.append(
            {
                "field": column,
                "value": value,
                "movie_count": int(count),
                "percent_of_movies": (
                    round(count / len(movies) * 100, 2) if len(movies) else pd.NA
                ),
            }
        )

category_profile = pd.DataFrame(category_rows)
display(category_profile)


,movie_count
release_date,
1966,1
1972,1
1974,1
1975,1
1977,1
1979,1
1980,1
1981,1
1982,1


,movie_count
release_date,
2004,6
2005,3
2006,5
2007,6
2008,4
2009,3
2010,5
2011,4
2012,3


,field,value,movie_count,percent_of_movies
0,status,Released,112,100.0000
1,original_language,en,110,98.2100
2,original_language,ja,1,0.8900
3,original_language,it,1,0.8900
4,adult,False,112,100.0000
5,video,False,112,100.0000


#### 3.1.2. Profile relationships

Count links per film, including films with no links. The genre reference check counts distinct genre IDs and separately shows how many film links they represent.


In [10]:
relationship_tables = {
    "genres": tables.movie_genres,
    "production companies": tables.movie_companies,
    "origin countries": tables.movie_origin_countries,
    "production countries": tables.movie_production_countries,
    "spoken languages": tables.movie_spoken_languages,
    "cast credits": tables.cast_credits,
    "crew credits": tables.crew_credits,
}

relationship_rows = []
for relationship_name, links in relationship_tables.items():
    counts_per_movie = links.groupby("movie_id").size().reindex(movies["movie_id"], fill_value=0)
    relationship_rows.append(
        {
            "relationship": relationship_name,
            "link_rows": len(links),
            "movies_with_at_least_one": int(counts_per_movie.gt(0).sum()),
            "movies_with_zero": int(counts_per_movie.eq(0).sum()),
            "mean_link_per_movie_including_zero": (
                float(counts_per_movie.mean()) if len(movies) else pd.NA
            ),
            "median_links_per_movie_including_zero": (
                float(counts_per_movie.median()) if len(movies) else pd.NA
            ),
            "maximum_links_for_one_movie": (
                int(counts_per_movie.max()) if len(movies) else pd.NA
            ),
        }
    )

relationship_profile = pd.DataFrame(relationship_rows)
display(relationship_profile)

genre_counts = (
    tables.movie_genres.groupby("genre_id")
    .size()
    .rename("movie_count")
    .reset_index()
    .merge(
        tables.genres,
        on="genre_id",
        how="left",
        validate="many_to_one",
        indicator="genre_match",
    )
    .sort_values("movie_count", ascending=False)
)
display(genre_counts.head(20))
print("Genre-reference matches:")
display(genre_counts.groupby("genre_match", observed=True).agg(
    distinct_genres=("genre_id", "size"),
    movie_genre_links=("movie_count", "sum"),
))


,relationship,link_rows,movies_with_at_least_one,movies_with_zero,mean_link_per_movie_including_zero,median_links_per_movie_including_zero,maximum_links_for_one_movie
0,genres,320,112,0,2.8571,3.0000,5
1,production companies,307,112,0,2.7411,3.0000,7
2,origin countries,117,112,0,1.0446,1.0000,2
3,production countries,154,112,0,1.3750,1.0000,4
4,spoken languages,219,112,0,1.9554,1.0000,6
5,cast credits,7827,112,0,69.8839,65.5000,177
6,crew credits,24929,112,0,222.5804,182.0000,953


,genre_id,movie_count,genre_name,genre_match
5,28,61,Action,both
0,12,60,Adventure,both
11,878,36,Science Fiction,both
9,53,33,Thriller,both
1,14,27,Fantasy,both
3,18,19,Drama,both
6,35,19,Comedy,both
10,80,18,Crime,both
2,16,17,Animation,both
14,10751,17,Family,both


,distinct_genres,movie_genre_links
genre_match,,
both,16,320


#### 3.1.3. Genre, collection, and talent coverage for dated releases

A film can belong to several genres. A film without a collection ID counts as a standalone. For each director or top-billed cast member, count their films and the distinct groups those films come from. All films in one collection count as one group. Each standalone is its own group.

In [11]:
# Use IDs for counts and bring in names for display.
genre_links = tables.movie_genres.loc[tables.movie_genres["movie_id"].isin(films["movie_id"])]
genre_counts = (genre_links.groupby("genre_id")["movie_id"].nunique()
                .rename("films").reset_index()
                .merge(tables.genres, on="genre_id", validate="one_to_one"))
print("Dated releases by genre")
display(genre_counts.sort_values(["films", "genre_id"], ascending=[False, True]))

print("Collection-linked and standalone releases")
display(films["collection_id"].notna().map({
    True: "Collection-linked", False: "Standalone",
}).value_counts().rename("films").to_frame())

raw_order = tables.cast_credits["order"]
cast_order = pd.to_numeric(raw_order.astype("string"), errors="coerce")
directors = tables.crew_credits.loc[
    tables.crew_credits["job"].eq("Director"), ["movie_id", "person_id"]
].assign(role="Director")
lead_cast = tables.cast_credits.loc[
    cast_order.isin([0, 1, 2]), ["movie_id", "person_id"]
].assign(role="Top-three cast")

# Keep collection and movie ID spaces distinct even if they use the same number.
film_groups = films[["movie_id", "collection_id"]].copy()
film_groups["is_standalone"] = film_groups["collection_id"].isna()
film_groups["group_id"] = film_groups["collection_id"].fillna(film_groups["movie_id"])
talent = (pd.concat([directors, lead_cast], ignore_index=True)
          .drop_duplicates(["role", "person_id", "movie_id"])
          .merge(film_groups, on="movie_id", validate="many_to_one"))
person_key = ["role", "person_id"]
title_counts = talent.groupby(person_key)["movie_id"].nunique().rename("films")
group_counts = (talent.drop_duplicates([*person_key, "is_standalone", "group_id"])
                .groupby(person_key).size().rename("distinct_groups"))
talent_coverage = (pd.concat([title_counts, group_counts], axis=1).reset_index()
                   .merge(tables.people[["person_id", "name"]], on="person_id", validate="many_to_one"))
talent_coverage["three_or_more_films"] = talent_coverage["films"].ge(3)
talent_coverage["three_films_across_groups"] = (
    talent_coverage["three_or_more_films"] & talent_coverage["distinct_groups"].ge(2)
)
print("Talent coverage by role")
display(talent_coverage.groupby("role").agg(
    people=("person_id", "size"),
    people_with_three_or_more_films=("three_or_more_films", "sum"),
    people_with_three_films_across_groups=("three_films_across_groups", "sum"),
))

,genre_id,films,genre_name
5,28,61,Action
0,12,60,Adventure
11,878,36,Science Fiction
9,53,33,Thriller
1,14,27,Fantasy
3,18,19,Drama
6,35,19,Comedy
10,80,18,Crime
2,16,17,Animation
14,10751,17,Family


,films
collection_id,
Collection-linked,96
Standalone,16


,people,people_with_three_or_more_films,people_with_three_films_across_groups
role,,,
Director,66,13,6
Top-three cast,184,44,12


### 3.2. Audience ratings, production budgets, and box-office gross by category

Compare the middle film in each group on reported production budget, historical box-office gross, and raw audience rating. Vote counts show how much audience feedback the ratings have. Each measure shows how many films supplied a usable value. Budget and gross are in reported historical dollars.

Genre rows cover genres with at least 10 dated releases. Collection rows use observed installments. Talent rows cover people with at least three observed films and show how many different groups those films came from. A film can contribute to multiple genres or people. These figures describe the films in each group and do not measure the effect of a genre or a person on an outcome. Production budgets and box-office grosses are separate from licensing fees.


In [12]:
def summarize_outcomes(group_films, keys):
    # Each group contains one row per film. Count usable values separately for each median.
    return group_films.groupby(keys).agg(
        films=("movie_id", "size"),
        films_with_budget=("budget", "count"),
        median_budget_dollars=("budget", "median"),
        films_with_gross=("revenue", "count"),
        median_gross_dollars=("revenue", "median"),
        films_with_rating=("vote_average", "count"),
        median_rating=("vote_average", "median"),
        films_with_vote_count=("vote_count", "count"),
        median_votes=("vote_count", "median"),
    ).reset_index()


outcome_fields = ["movie_id", "budget", "revenue", "vote_average", "vote_count"]
genre_films = genre_links.merge(
    films[outcome_fields], on="movie_id", validate="many_to_one",
)
genre_outcomes = summarize_outcomes(genre_films, ["genre_id"]).merge(
    tables.genres, on="genre_id", validate="one_to_one",
)
genre_outcomes = genre_outcomes.loc[genre_outcomes["films"].ge(10)]
if genre_outcomes.empty:
    print("No genre has at least 10 dated releases.")
else:
    print("Outcomes by genre")
    display(genre_outcomes.sort_values(["films", "genre_id"], ascending=[False, True])
            .set_index(["genre_id", "genre_name"]))

# Compare film-level outcomes for both types before looking at individual collections.
film_types = films.assign(film_type=films["collection_id"].notna().map({
    True: "Collection-linked", False: "Standalone",
}))
type_outcomes = summarize_outcomes(film_types, ["film_type"])
print("Outcomes by collection membership")
display(type_outcomes.set_index("film_type"))

collection_outcomes = summarize_outcomes(films, ["collection_id"]).merge(
    tables.collections, on="collection_id", validate="one_to_one",
)
print("Outcomes per observed collection")
display(collection_outcomes.sort_values(["films", "collection_id"], ascending=[False, True])
        .set_index(["collection_id", "collection_name"]))

# Each film counts once per person and role, even if a person has several credits.
talent_films = talent.merge(films[outcome_fields], on="movie_id", validate="many_to_one")
talent_outcomes = summarize_outcomes(talent_films, person_key).merge(
    talent_coverage[[*person_key, "name", "distinct_groups"]],
    on=person_key, validate="one_to_one",
)
talent_outcomes = talent_outcomes.loc[talent_outcomes["films"].ge(3)]
print("Outcomes for directors and top-billed cast with at least three films")
display(talent_outcomes.sort_values(["role", "distinct_groups", "films", "person_id"],
                                   ascending=[True, False, False, True])
        .set_index(["role", "person_id", "name", "distinct_groups"]))


,,films,films_with_budget,median_budget_dollars,films_with_gross,median_gross_dollars,films_with_rating,median_rating,films_with_vote_count,median_votes
genre_id,genre_name,,,,,,,,,
28,Action,61,61,"115,000,000.0000",61,"585,174,222.0000",61,7.4500,61,"15,440.0000"
12,Adventure,60,60,"125,000,000.0000",60,"781,017,020.0000",60,7.6105,60,"17,712.5000"
878,Science Fiction,36,36,"107,500,000.0000",36,"529,855,930.0000",36,7.7415,36,"17,357.0000"
53,Thriller,33,33,"85,000,000.0000",33,"366,101,666.0000",33,7.4500,33,"11,335.0000"
14,Fantasy,27,27,"150,000,000.0000",27,"926,287,400.0000",27,7.6820,27,"20,350.0000"
18,Drama,19,19,"40,000,000.0000",19,"245,066,411.0000",19,8.2500,19,"18,194.0000"
35,Comedy,19,19,"80,000,000.0000",19,"563,749,323.0000",19,7.5000,19,"14,528.0000"
80,Crime,18,18,"42,000,000.0000",18,"258,904,666.5000",18,8.1570,18,"17,901.0000"
16,Animation,17,17,"92,000,000.0000",17,"650,000,000.0000",17,7.6230,17,"15,594.0000"


,films,films_with_budget,median_budget_dollars,films_with_gross,median_gross_dollars,films_with_rating,median_rating,films_with_vote_count,median_votes
film_type,,,,,,,,,
Collection-linked,96,96,"101,000,000.0000",96,"578,937,111.0000",96,7.6010,96,"15,482.5000"
Standalone,16,16,"25,000,000.0000",16,"161,802,536.5000",16,8.4255,16,"18,854.5000"


,,films,films_with_budget,median_budget_dollars,films_with_gross,median_gross_dollars,films_with_rating,median_rating,films_with_vote_count,median_votes,collection_poster_path,collection_backdrop_path
collection_id,collection_name,,,,,,,,,,,
1241,Harry Potter Collection,8,8,"140,000,000.0000",8,"936,085,967.5000",8,7.7680,8,"22,850.5000",/eVPs2Y0LyvTLZn6AP5Z6O2rtiGB.jpg,/4gV0rKUjB1nLUdZB4zIltLvNZZr.jpg
10,Star Wars Collection,6,6,"72,675,000.0000",6,"712,398,167.5000",6,7.6895,6,"17,000.5000",/pWVLFh4OuejTpUaDQbB1C4zoS2p.jpg,/iY2ujEY2m68OTTlPFTiHub9joHS.jpg
295,Pirates of the Caribbean Collection,5,5,"230,000,000.0000",5,"961,691,209.0000",5,7.2810,5,"16,072.0000",/zRBaZxS5YauLvRYjAdL4AUCwlht.jpg,/wxgD3fB5lQ2sGJLog0rvXW049Pf.jpg
87359,Mission: Impossible Collection,5,5,"145,000,000.0000",5,"546,388,108.0000",5,7.0360,5,"10,106.0000",/geEjCGfdmRAA1skBPwojcdvnZ8A.jpg,/mroWh717g0Ah2c0rrPGW6f3EWMM.jpg
84,Indiana Jones Collection,4,4,"38,000,000.0000",4,"432,048,888.5000",4,7.6045,4,"11,037.0000",/lpxDrACKJhbbGOlwVMNz5YCj6SI.jpg,/6kh59mZizZsttZPR0HAdXk6Ve2n.jpg
528,The Terminator Collection,4,4,"151,000,000.0000",4,"403,176,500.5000",4,6.9575,4,"11,421.0000",/kpZxdNsAV7qTdTLwKM5NLqa7GEo.jpg,/sCnBEw2Yu6foEjs4Xb4eMddYHRo.jpg
1570,Die Hard Collection,4,4,"80,000,000.0000",4,"303,066,380.0000",4,7.1300,4,"6,840.0000",/xhnb5lVfwE7NHycdPNdIxHx7kZi.jpg,/nKShbIW0wJnSbl3t8uPje0QhTb7.jpg
2150,Shrek Collection,4,4,"155,000,000.0000",4,"780,455,527.0000",4,6.8745,4,"12,060.0000",/g1Kcbi5zSelArzR5UYkxKvG6K2a.jpg,/lhsd1zCsq5UquvcNalmhuddV3tI.jpg
8091,Alien Collection,4,4,"34,250,000.0000",4,"160,886,772.5000",4,7.2000,4,"9,055.0000",/gWFHIY77cRVoBRGERwMHqpD27gc.jpg,/6X42JnSMdo3dPAswOHUuvebdTq7.jpg


films  \
role           person_id name                  distinct_groups          
Director       525       Christopher Nolan     5                    7   
               488       Steven Spielberg      4                    8   
               108       Peter Jackson         2                    6   
               24        Robert Zemeckis       2                    4   
               138       Quentin Tarantino     2                    3   
               2710      James Cameron         2                    3   
               1         George Lucas          1                    4   
               11343     David Yates           1                    4   
               1704      Gore Verbinski        1                    3   
               1776      Francis Ford Coppola  1                    3   
               1884      Steven Soderbergh     1                    3   
               9339      Lilly Wachowski       1                    3   
               9340      Lana Wachowski        1                    3   
Top-three cast 3         Harrison Ford         3                    8   
               1158      Al Pacino             3                    5   
               3894      Christian Bale        3                    5   
               3895      Michael Caine         3                    3   
               3896      Liam Neeson           3                    3   
               3223      Robert Downey Jr.     2                    7   
               1327      Ian McKellen          2                    6   
               31        Tom Hanks             2                    5   
               287       Brad Pitt             2                    4   
               530       Carrie-Anne Moss      2                    4   
               1892      Matt Damon            2                    4   
               139       Uma Thurman           2                    3   
               10980     Daniel Radcliffe      1                    8   
               10989     Rupert Grint          1                    8   
               10990     Emma Watson           1                    8   
               85        Johnny Depp           1                    5   
               500       Tom Cruise            1                    5   
               62        Bruce Willis          1                    4   
               118       Geoffrey Rush         1                    4   
               776       Eddie Murphy          1                    4   
               6941      Cameron Diaz          1                    4   
               10205     Sigourney Weaver      1                    4   
               12073     Mike Myers            1                    4   
               12898     Tim Allen             1                    4   
               2         Mark Hamill           1                    3   
               4         Carrie Fisher         1                    3   
               103       Mark Ruffalo          1                    3   
               109       Elijah Wood           1                    3   
               110       Viggo Mortensen       1                    3   
               114       Orlando Bloom         1                    3   
               521       Michael J. Fox        1                    3   
               524       Natalie Portman       1                    3   
               1062      Christopher Lloyd     1                    3   
               1100      Arnold Schwarzenegger 1                    3   
               1461      George Clooney        1                    3   
               2975      Laurence Fishburne    1                    3   
               3061      Ewan McGregor         1                    3   
               5723      John Leguizamo        1                    3   
               5724      Denis Leary           1                    3   
               6384      Keanu Reeves          1                    3   
               7060      Martin Freeman        1 

### 3.3. Does genre help explain reported gross?

Compare a model using production budget, release year, and collection status with the same model plus genre tags. A tag needs at least 15 dated releases with usable budget, gross, and year for this model. Test both models on the same held-out films using the same five splits. Gross and budget use a compressed scale inside the model so the largest values have less influence.

Higher R² and lower average error indicate a better fit on the films held out of training. Films from one collection can appear on both sides of a split here, which limits what the result says about a new collection. Notebook 04 checks the genres in both windows while keeping related films together.

In [13]:
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import KFold, cross_validate

# Fit both versions on the same films so any difference comes from adding the tags.
model_films = films.dropna(subset=["budget", "revenue", "release_year"]).set_index("movie_id")
genre_flags = pd.crosstab(genre_links["movie_id"], genre_links["genre_id"])
genre_flags = genre_flags.reindex(index=model_films.index, fill_value=0)
common_genres = genre_flags.loc[:, genre_flags.sum().ge(15)]
if len(model_films) < 10 or common_genres.empty:
    print("Too few films or common tags to compare the two models.")
else:
    controls = pd.DataFrame({
        "log_budget": np.log10(model_films["budget"]),
        "release_year": model_films["release_year"],
        "has_collection": model_films["collection_id"].notna(),
    })
    target = np.log10(model_films["revenue"])
    folds = KFold(n_splits=5, shuffle=True, random_state=42)
    model_results = []
    for label, features in (
        ("Budget, year, and collection status", controls),
        ("Same fields plus genre", controls.join(common_genres, validate="one_to_one")),
    ):
        scores = cross_validate(
            LinearRegression(), features.to_numpy(dtype=float), target, cv=folds,
            scoring={"r2": "r2", "error": "neg_mean_absolute_error"},
        )
        model_results.append({
            "Model": label,
            "Films": len(model_films),
            "Held-out R²": scores["test_r2"].mean(),
            "Average held-out error (log gross)": -scores["test_error"].mean(),
        })
    display(pd.DataFrame(model_results).round(4))

,Model,Films,Held-out R²,Average held-out error (log gross)
0,"Budget, year, and collection status",112,0.5019,0.2116
1,Same fields plus genre,112,0.4996,0.2086


## 4. Findings and limits

### 4.1. Findings from the 112-film snapshot

- **Unusual values can be real.** The high box-office gross of Avengers: Endgame and the 202-minute runtime of The Godfather Part II are plausible. Keep these films in the main analysis and check how much the results change when flagged films are left out. Treat invalid values, such as negative gross, separately.
- **Bigger production budgets generally came with higher box-office grosses.** Pearson correlation was 0.746 across 112 films and 0.715 across the 109 left after leaving out unusually high budgets or grosses. The pattern persists without those films. This describes theatrical scale. The figures do not tell us what the films cost to license.
- **Popularity and vote count overlap strongly (0.830).** Gross and rating had very little straight-line relationship (−0.021). The two measures tell us about different outcomes and neither measures streaming performance.
- **Adding genre tags gave mixed results on held-out films.** R² changed from about 0.502 to 0.500 and average log-gross error changed from 0.212 to 0.209.
- **Of the 13 directors and 44 top-three cast members with three or more films, only 6 directors and 12 cast members had credits across two or more groups**. Repeated credits in one series provide limited evidence about a person's wider track record.

### 4.2. Limits of the evidence

These films are a selected list. The collection counts cover observed installments and the credits cover only the films in this list. Use Notebook 04 for the two-window recommendation, the shortlists, and the test of whether talent results hold up after leaving out a group. Historical gross describes theatrical performance. Licensing prices and streaming outcomes are unavailable here.